# 🎥 ROTBOTS — Video Generator
## From Prompts to Video Clips

---

This notebook takes the `prompts.json` from the Script Writer and generates video clips.

1. **Generate** — Send each prompt to fal.ai to create video clips
2. **Found Footage** — Optionally upload your own clips
3. **Preview** — Watch and compare all clips

> **🤔 Think about:** What does it cost (financially, environmentally) to generate these clips?

---

## 🔧 Setup

In [ ]:
# ============================================================
# SETUP — Run this cell first!
# ============================================================

!pip install -q fal-client requests

import fal_client
import json
import os
import time
import requests
from pathlib import Path
from IPython.display import display, Markdown, HTML, Video

# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
WORK_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR = WORK_DIR / 'videos'
VIDEOS_DIR.mkdir(exist_ok=True)

print(f'✅ Connected to Google Drive')
print(f'📁 Shared workspace: {WORK_DIR}')
print()
print('Files from previous notebooks:')
for f in sorted(WORK_DIR.glob('*.json')):
    print(f'   ✅ {f.name} ({f.stat().st_size / 1024:.1f}KB)')

In [ ]:
# ============================================================
# API KEY — Paste your fal.ai API key here
# Get one at: https://fal.ai/dashboard/keys
# ============================================================

FAL_API_KEY = ""  # <-- Paste your fal.ai key here

if not FAL_API_KEY:
    print("⚠️  Please paste your fal.ai API key above!")
    print("   Get one at: https://fal.ai/dashboard/keys")
else:
    os.environ['FAL_KEY'] = FAL_API_KEY
    print('✅ fal.ai API key configured')

In [ ]:
# ============================================================
# LOAD PROMPTS from Google Drive
# ============================================================

prompts_file = WORK_DIR / 'prompts.json'

if prompts_file.exists():
    with open(prompts_file) as f:
        prompts = json.load(f)
    print(f'✅ Loaded {len(prompts)} prompts from Google Drive')
    for p in prompts:
        print(f"   Scene {p['scene']}: {p['title']} [{p['style']}] ({p['duration']}s)")
else:
    print('❌ No prompts.json found!')
    print('   Run the Script Writer notebook (02_Script_Writer.ipynb) first.')
    print(f'   Expected: {prompts_file}')

---
## 🎬 Station 5: Generate Videos

Each clip takes **1-5 minutes** to generate.

> **🤔 Discussion:** Each clip costs $0.25-$0.50. Who owns this generated footage?

In [ ]:
# ============================================================
# VIDEO MODEL SELECTION
# ============================================================

VIDEO_MODELS = {
    "wan-t2v": {
        "endpoint": "fal-ai/wan-t2v",
        "description": "Wan 2.1 T2V — Good quality, affordable (~$0.10/s)",
        "default_seconds": 5,
    },
    "wan-2.2": {
        "endpoint": "fal-ai/wan/v2.2-a14b/text-to-video",
        "description": "Wan 2.2 A14B — Best open-source quality ($0.10/s)",
        "default_seconds": 5,
    },
    "minimax": {
        "endpoint": "fal-ai/minimax/video-01",
        "description": "MiniMax Hailuo 01 — High quality (~$0.28/video)",
        "default_seconds": 6,
    },
    "ltx-video": {
        "endpoint": "fal-ai/ltx-video/v2.1",
        "description": "LTX Video 2.1 — Fast generation",
        "default_seconds": 5,
    },
}

# ⬇️ CHOOSE YOUR MODEL HERE ⬇️
CHOSEN_MODEL = "wan-t2v"  # Options: wan-t2v, wan-2.2, minimax, ltx-video

model = VIDEO_MODELS[CHOSEN_MODEL]
print(f"🎥 Model: {CHOSEN_MODEL} — {model['description']}")
print(f"   Endpoint: {model['endpoint']}")
print(f"   Estimated cost for {len(prompts)} scenes: ~${len(prompts) * 0.30:.2f} - ${len(prompts) * 0.50:.2f}")

In [ ]:
# ============================================================
# GENERATE VIDEOS — Takes a few minutes per scene!
# ============================================================

print(f"🎬 Generating {len(prompts)} clips with {CHOSEN_MODEL}...")
print(f"   Estimated time: {len(prompts) * 2}-{len(prompts) * 5} minutes")
print('=' * 60)

generated_videos = []

for i, prompt_data in enumerate(prompts):
    scene_num = prompt_data['scene']
    t2v_prompt = prompt_data['t2v_prompt']
    duration = prompt_data.get('duration', model['default_seconds'])

    print(f"\n🎥 Scene {scene_num} ({i+1}/{len(prompts)}): {prompt_data['title']}")
    print(f"   Prompt: {t2v_prompt[:100]}...")
    print(f"   Generating ({duration}s)...", end=' ', flush=True)

    start_time = time.time()
    try:
        input_data = {'prompt': t2v_prompt}

        # Model-specific parameters
        if 'wan' in CHOSEN_MODEL:
            input_data['aspect_ratio'] = '16:9'
            input_data['enable_prompt_expansion'] = False
        elif CHOSEN_MODEL == 'ltx-video':
            input_data['aspect_ratio'] = '16:9'

        result = fal_client.subscribe(model['endpoint'], arguments=input_data)
        elapsed = time.time() - start_time

        # Extract video URL (different models return it differently)
        video_url = None
        if isinstance(result, dict):
            if 'video' in result and isinstance(result['video'], dict):
                video_url = result['video'].get('url')
            elif 'video' in result and isinstance(result['video'], str):
                video_url = result['video']
            elif 'url' in result:
                video_url = result['url']
            elif 'output' in result:
                video_url = result['output'].get('url') if isinstance(result['output'], dict) else result['output']

        if video_url:
            video_path = VIDEOS_DIR / f'scene_{scene_num:03d}.mp4'
            resp = requests.get(video_url, timeout=120)
            with open(video_path, 'wb') as f:
                f.write(resp.content)
            size_kb = video_path.stat().st_size / 1024
            generated_videos.append({'scene': scene_num, 'title': prompt_data['title'], 'path': str(video_path), 'duration': duration, 'source': 'generated'})
            print(f'✅ Done! ({elapsed:.0f}s, {size_kb:.0f}KB)')
        else:
            print(f'⚠️ No video URL found in response')
            print(f'   Keys: {list(result.keys()) if isinstance(result, dict) else type(result)}')

    except Exception as e:
        elapsed = time.time() - start_time
        print(f'❌ Error after {elapsed:.0f}s: {e}')

print(f"\n{'='*60}")
print(f'✅ Generated {len(generated_videos)}/{len(prompts)} videos')
print(f'📁 Saved to: {VIDEOS_DIR}')

---
## 📂 Found Footage (Optional)

Upload your own MP4 clips to mix with or replace AI-generated ones.

In [ ]:
# UPLOAD FOUND FOOTAGE (Optional)
USE_FOUND_FOOTAGE = False  # Set to True to upload

if USE_FOUND_FOOTAGE:
    from google.colab import files
    print('📂 Upload MP4 files:')
    uploaded = files.upload()
    for filename, content in uploaded.items():
        dest = VIDEOS_DIR / filename
        with open(dest, 'wb') as f: f.write(content)
        scene_num = int(''.join(filter(str.isdigit, filename.split('.')[0]))) if any(c.isdigit() for c in filename) else 100 + len(generated_videos)
        generated_videos.append({'scene': scene_num, 'title': f'Found: {filename}', 'path': str(dest), 'duration': 0, 'source': 'found_footage'})
        print(f'   ✅ {filename} ({len(content)/1024:.0f}KB)')
else:
    print('ℹ️ Set USE_FOUND_FOOTAGE = True above to upload your own clips.')

---
## 👀 Preview Videos

In [ ]:
# PREVIEW ALL VIDEOS
generated_videos.sort(key=lambda x: x['scene'])
print(f'🎬 {len(generated_videos)} video clips:')
for v in generated_videos:
    tag = '🤖 AI' if v['source'] == 'generated' else '📂 Found'
    display(Markdown(f"### Scene {v['scene']}: {v['title']} — {tag}"))
    if Path(v['path']).exists():
        try: display(Video(v['path'], embed=True, width=640))
        except: print(f'   (File at: {v["path"]})')
    display(Markdown('---'))

In [ ]:
# SAVE MANIFEST
manifest = {'model': CHOSEN_MODEL, 'videos': generated_videos}
with open(WORK_DIR / 'video_manifest.json', 'w') as f: json.dump(manifest, f, indent=2)
print(f'✅ Manifest saved ({len(generated_videos)} clips)')
print(f'\n📁 Videos on Drive:')
for f in sorted(VIDEOS_DIR.glob('*.mp4')): print(f'   {f.name} ({f.stat().st_size/1024:.0f}KB)')
print('\n→ Next: 04_The_Voice.ipynb for narration, then 06_Assemble.ipynb!')

---
## ⏭️ Next Steps

Videos saved on Google Drive. Next:
- **04_The_Voice.ipynb** — Generate narration audio
- **06_Assemble.ipynb** — Combine videos + audio into final video

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026*